In [5]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display

In [6]:
load_dotenv()
os.getenv("GROQ_API_KEY")  # Ensure the API key
openai = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")
gemini = OpenAI(api_key=os.getenv("GOOGLE_API_KEY"), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [7]:
OPENAI_MODEL = "openai/gpt-oss-120b"
GEMINI_MODEL = "gemini/gemini-3.1-flash-lite"

In [8]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Linux',
  'arch': 'x86_64',
  'release': '7.2.5-3-omarchy',
  'version': '#1 SMP PREEMPT_DYNAMIC Mon, 14 Sep 2026 19:55:01 +0000',
  'kernel': '7.2.5-3-omarchy',
  'distro': {'name': 'Omarchy', 'version': '4.0.4'},
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'x86_64-pc-linux-gnu'},
 'package_managers': ['pacman'],
 'cpu': {'brand': '11th Gen Intel(R) Core(TM) i5-1145G7 @ 2.60GHz',
  'cores_logical': 8,
  'cores_physical': 4,
  'simd': ['AVX', 'AVX2', 'AVX512F', 'FMA', 'SSE4_2']},
 'toolchain': {'compilers': {'gcc': 'gcc (GCC) 16.2.1 20260810',
   'g++': 'g++ (GCC) 16.2.1 20260810',
   'clang': 'clang version 22.1.8',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 4.4.1'},
  'linkers': {'ld_lld': ''}}}

In [9]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model=OPENAI_MODEL, messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))


## 1️⃣ Do you need to install a C++ compiler?

The system information you posted already shows that a **full C++ tool‑chain is present**:

| Tool | Version |
|------|---------|
| `gcc` (C) | 16.2.1 |
| `g++` (C++) | 16.2.1 |
| `clang` | 22.1.8 |
| `make` | 4.4.1 |

Because `g++` (the GNU C++ compiler) is installed, you can already compile and run a C++ file. **No additional installation is required**.

> **If you ever find yourself on a machine where the compilers are missing**, the quickest way on an Arch‑based distro (your system uses `pacman`) would be:

```bash
sudo pacman -Sy   # update the package database
sudo pacman -S gcc clang make   # install the compilers and make
```

But for the current machine you can go straight to the compile‑and‑run step.

---

## 2️⃣ “Fastest possible runtime performance” – which flags should we use?

For a single source file, the most aggressive, yet still portable, optimization flags are:

* `-O3` – highest level of optimisation that the compiler provides.
* `-march=native` – let the compiler generate instructions tuned for **your exact CPU** (Intel 11th‑gen i5 in this case). This enables AVX/AVX2/AVX‑512, FMA, etc.
* `-flto` – link‑time optimisation – the compiler can see the whole program and inline across translation units (even though you have only one file, it still enables more aggressive optimisations).
* `-std=c++23` – use the newest language features; you can drop this if you only need an older standard.
* `-pipe` – use pipes rather than temporary files (a tiny speed‑up during compilation).
* `-s` – strip the binary of symbol tables/debug info (makes the executable smaller; does **not** affect runtime speed).

Putting it together, the compile command is:

```bash
g++ -std=c++23 -O3 -march=native -flto -pipe -s main.cpp -o main
```

(If you prefer `clang++`, just replace `g++` with `clang++` and keep the same flags; clang also understands `-march=native` and `-flto`.)

---

## 3️⃣ Python snippet that compiles **and then runs** the program

Below is a minimal, self‑contained Python function that:

1. Calls the compile command shown above.
2. Executes the resulting binary (`./main`).
3. Returns the program’s standard output.

```python
import subprocess
from pathlib import Path

def compile_and_run_cpp(source_path: Path = Path("main.cpp"),
                        exe_path: Path = Path("main")) -> str:
    """
    Compiles `source_path` with g++ (fastest‑runtime flags) and runs the
    resulting executable, returning its stdout as a string.
    Raises subprocess.CalledProcessError on any failure.
    """
    # ------------------------------------------------------------------
    # 1️⃣ Compile
    # ------------------------------------------------------------------
    compile_cmd = [
        "g++",                     # compiler
        "-std=c++23",              # language version
        "-O3",                     # highest optimisation level
        "-march=native",           # tune for this CPU
        "-flto",                   # link‑time optimisation
        "-pipe",                   # use pipes instead of temp files
        "-s",                      # strip symbols (smaller binary)
        str(source_path),          # source file
        "-o", str(exe_path)        # output executable
    ]

    compile_result = subprocess.run(
        compile_cmd,
        check=True,                # raise if compilation fails
        text=True,
        capture_output=True        # capture gcc/clang messages (stderr)
    )
    # If you want to see any compiler warnings/errors:
    # print("Compiler stdout:", compile_result.stdout)
    # print("Compiler stderr:", compile_result.stderr)

    # ------------------------------------------------------------------
    # 2️⃣ Run the compiled program
    # ------------------------------------------------------------------
    run_cmd = [str(exe_path)]

    run_result = subprocess.run(
        run_cmd,
        check=True,                # raise if the program exits non‑zero
        text=True,
        capture_output=True        # capture the program's stdout
    )

    return run_result.stdout

# ----------------------------------------------------------------------
# Example usage
# ----------------------------------------------------------------------
if __name__ == "__main__":
    output = compile_and_run_cpp()
    print("Program output:\n", output)
```

### What the code does, line‑by‑line

| Section | Explanation |
|---------|-------------|
| `compile_cmd` list | Exact command that `subprocess.run` will execute. Each list element becomes a separate argument – no shell quoting needed. |
| `subprocess.run(..., check=True, ...)` | Throws `CalledProcessError` if the compiler returns a non‑zero exit status, so you know immediately if compilation failed. |
| `capture_output=True` | Captures both `stdout` and `stderr` in the `CompletedProcess` object (`compile_result.stdout`, `compile_result.stderr`). Useful for debugging. |
| `run_cmd = [str(exe_path)]` | Runs the freshly built binary (`./main`). |
| `return run_result.stdout` | Gives you the program’s printed output as a Python `str`. |

**Tip:** If you plan to compile many times in a row, you can add `-c` (compile only) and then link once, but for a single‑file workflow the command above is the simplest and fastest overall.

---

## 4️⃣ Quick “one‑liner” you could paste into a Python REPL

If you don’t want the full function and just need a quick test, the following two lines do the same thing:

```python
import subprocess, pathlib
src = pathlib.Path("main.cpp")
exe = pathlib.Path("main")
subprocess.run(["g++","-std=c++23","-O3","-march=native","-flto","-pipe","-s",str(src),"-o",str(exe)], check=True)
out = subprocess.run([str(exe)], capture_output=True, text=True, check=True).stdout
print(out)
```

---

### TL;DR

* **You’re already set up** – `g++` and `clang++` are installed.
* Use the compile command  
  ```bash
  g++ -std=c++23 -O3 -march=native -flto -pipe -s main.cpp -o main
  ```
* Run the program with  
  ```bash
  ./main
  ```
* In Python, feed those exact argument lists to `subprocess.run` as shown above.

That’s all you need to compile a single `main.cpp` file and execute it with the fastest runtime performance the current hardware can provide. Happy coding! 🚀

In [10]:
compile_command = ["clang++", "-std=c++23", "-O3", "-march=native", "-flto", "-pipe", "-s", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [11]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
"""

In [12]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]


In [13]:
def write_output(cpp):
    with open("main.cpp", "w", encoding="utf-8") as f:
        f.write(cpp)

In [14]:
def port(client, model, python):
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(reply)

In [15]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [16]:
def run_python(code):
    globals = {"__builtins__": __builtins__}
    exec(code, globals)

In [17]:
run_python(pi)

Result: 3.141592656089
Execution Time: 20.235190 seconds


In [20]:
port(openai, OPENAI_MODEL, pi)